# Aim
Create a `LangChain` workflow which orchestrates two LLMs - the first one is the one already implemented in `llm_geolocations.ipynb`, the other one is a subsequent model for extracting impact information based on the output (CI_type, damage_type, geolocation) of the first LLM 

In [ ]:
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=0  # nvidia gpu
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
# %env TORCH_CUDA_ARCH_LIST=8.6

# settings for distributed computing
%env WORLD_SIZE=1
%env RANK=0
%env LOCAL_RANK=0

# NOTE: # WORLD_SIZE: each GPU corresponds to one process (world = no. of processes within a group), processes communicate with each other enabling eg., distributed training
# NOTE: # RANK: IDs of the processes, ranging from 0 up to WORLD_SIZE - 1

In [3]:
import os
import sys
import re
import glob
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
from jinja2 import Template
from langchain_docling import DoclingLoader
from huggingface_hub import login

import torch
import transformers


sys.path.append("../")
import src.settings as s

torch.manual_seed(42)

# set default location to store model before loading transformers
os.environ["HF_HOME"] = (
    "/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/"
)

/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
OUTPUT_DIR = "../" + s.settings.PATH_DATA + "llm_outputs/"


### Load response from first LLM

In [9]:

outfile_name = "ci_failure_impacts_responses_llama2_with_eco_explicit_3.csv"
outfile_path = OUTPUT_DIR +  outfile_name


with open(outfile_path, "r") as f:
    df_llm1 = pd.read_csv(f)

df_llm1.tail()

,chunk_id,infrastructure_type,damage,societal_impact,economic_impact,location,confidence,confidence_explanation,citation
23,8,solid-waste facilities,NAN,NAN,NAN,Rhineland-Palatinate,3,No information found on direct impact on solid...,Koks et al 2022
24,8,highway,160 000 t,Oil pollution,Estimated around 160 000 t,Belgium,4,Used for approximately 9 months as a temporary...,Koks et al 2022
25,8,solid-waste,Thousands of tonnes of tree debris,Problems with waste deposits along the river b...,NAN,Netherlands,3,Mostly the solid waste transported by the rive...,Koks et al 2022
26,9,hospital,"approximately 68 hospitals have been affected,...",direct damages are estimated to be at least EU...,property damage is expected to be around EUR 5...,"North Rhine-Westphalia, Germany",4,based on the information provided in the conte...,Koks et al 2022
27,9,power supply,the power supply collapsed,the entire building technology was destroyed,some 300 patients had to be evacuated by helic...,"Eschweiler, Germany",5,based on the information provided in the conte...,Koks et al 2022


###  Prompt engineering + TODO Adapt output struct

* input is response from LLM 1 
* output schema adapted to table structure of the evaluation table with manually extracted CI damages and impacts


In [ ]:

## with s+e impacts
# question = "Which impacts of infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, societal or economic impacts, the location and possibly the time of the infrastructure failure."

# ## without s+e impacts
# question = "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location and possibly the time of the infrastructure failure."

In [ ]:
prompt_template = """

    You are an expert analyst assistant and should use ONLY the provided context to answer the following question:
    
    Question: "{{ question }}"
       
    Context:
    {% for item in context %}
    - {{ item.text }} (Citation: {{ item.citation }})
    {% endfor %}


    For the the fields "societal_impact" and "economic_impact" you should try to extract information about societal or economic consequences of infrastructure failures mentioned in the context.
    Extract only the impacts and their location (field: "impact_location") for infrastructure assets (column: "infrastructure_type") which are mentioned in df_llm1.
    If you do not find any information about societal or economic consequences, then return for these fields a "NAN" value.

    For the field "impact_to_other_infrastructure" you should try to extract information about cascading impacts to other infrastructure assets, such as a disrupted broadband connection due to power blackout.
    If you do not find any information about these cascading consequences, then return for this field a "NAN" value.
    Try also to extract information about the location of the additionally affected infrastructure (field: "impact_to_other_infrastructure_location"), such as the location of the disrupted broadband connection.
    If you do not find any information about the location of these cascading consequences, then return for this field a "NAN" value.
    
    df_llm1:
    {% for item in context %}
    - ("ci_entity", "damage_type", and "geo_entity":\n {{ item["ci_locations"][["ci_entity", "damage_type", "geo_entity"]] }})
    {% endfor %}



    Return ONLY valid JSON in the following list format:
    [{
        "infrastructure_type": "...",
        "damage": "...",
        "societal_impact": "...",
        "societal_impact_location": "...",
        "economic_impact": "...",
        "economic_impact_location": "...",
        "impact_to_other_infrastructure": "...",
        "impact_to_other_infrastructure_location": "...",
    }]

    Each nested dictionary describes one failure case.
    DO NOT add commentary or text outside the JSON.
    

    Answer:
"""
# example_context = doc[6].page_content
# chunk_id = 6
# context = [
#     {
#         "text": example_context,
#         "citation": "Koks et al., 2022",
#         "ci_locations": df_ci_geo.loc[df_ci_geo["chunk_id"]==chunk_id],#to_dict(orient="records")
#     },  # TODO use author names or Primary keys from DB
#     # {"text": context, "citation": "Meier et al., 2025"},
# ]

# rendered_prompt = template.render(
#     context=context,
#     question=question,
#     # messages=messages
# )
# print(rendered_prompt)


## left overs
#  Try to be as specific as possible in your answer (bullet points), mention the impacts as numerical information along the location of the impact, and refer to the citations provided in the context.
# # Extract information about infrastructure failures based on the following question:

# impl Langchain workflow

# Apply LLama-2 on chunks 
* run new model to extract societal and economic impacts guided by the Ci-types and their locations returned from LLM 1


In [ ]:
# # empty CUDA cache
import gc
import torch

gc.collect()

torch.cuda.empty_cache()
# print(torch.cuda.memory_summary(device=None, abbreviated=False))

In [ ]:
# init class for decoder and tokenizer


class DecoderModel:
    def __init__(self):
        login(
            token=os.environ["HUGGINGFACE_TOKEN"]
        )  # TODO replace by using pydantic settings

        # model_name = "google/gemma-3-4b-it" # "kallidavidson/TinyBERT_General_4L_312D"  # "huawei-noah/TinyBERT_General_4L_312D" # - for QA - less DWL
        model_name = "meta-llama/Llama-2-7b-chat-hf"
        # model_name = "EleutherAI/gpt-j-6B" #"distilbert-base-multilingual-cased"
        base_dir = "./huggingface_mirror"  # use default dir in .cache/
        model_dir = base_dir + "/hub/"  # + "models--" + model_name.replace("/", "--")
        print(model_dir)

        # quantization config
        # Load model with 4-bit quantization if applicable (use 4-bit integer instead of 32b floats) --> reduce the required VRAM for model application
        # see, https://huggingface.co/docs/transformers/quantization
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )

        self.pipeline, self.tokenizer = self.initialize_model(
            model_name, model_dir, bnb_config
        )

    def initialize_model(self, model_name: str, model_dir: str = None, bnb_config=None):
        # Model and Tokenizer initialization
        if not os.path.exists(model_dir):
            print("Model directory not found. Downloading model...")
            os.makedirs(model_dir, exist_ok=True)

            device = transformers.infer_device()
            print(f"Using device: {device}")
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                dtype="auto",
                attn_implementation="flash_attention_2",  # use with 4-bit quantization,
                # --> flash attention enables to use much larger sequence lengths without running into OOM issues
                quantization_config=bnb_config,
                # max_memory={0: "2GB", 1: "10GB"},  # distribute memory across GPUs
            )
            model.save_pretrained(model_dir)
            tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
            tokenizer.save_pretrained(model_dir)

            print("Downloaded model and tokenizer")

        else:
            print(f"Using locally saved model from {model_dir}")

            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                cache_dir=model_dir,
                local_files_only=True,  # tp_plan="auto" # set tensor parallel model (ie. splits model on multiple GPU)
                # dtype="auto",
                attn_implementation="flash_attention_2",  # use with 4-bit quantization,
                # --> flash attention enables to use much larger sequence lengths without running into OOM issues
                quantization_config=bnb_config,
                # tp_plan="auto",  # automatically use a tensor parallelism plan based on predefined configuration of the model (i.e. partition model on both GPUs)
            )
            # print("Tensor parallel plan:", model._tp_plan)

            tokenizer = AutoTokenizer.from_pretrained(
                model_name,
                use_fast=True,
                cache_dir=model_dir,  # use fast Rust-based tokenizer, when possible
            )

        # reduce further memory usage
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)
        model.use_checkpointing = True

        torch.cuda.empty_cache()

        # Pipeline setup for question answering
        pipeline = transformers.pipeline(  # load model locally from wsl .cache\
            "text-generation",
            # "question-answering",  # task defining which pipeline is returned
            model=model,
            tokenizer=tokenizer,
            # (return_tensors="pt"),  # load specific tokenizer based on model-name (via AutoTokenizer) ensuring text is tokenized in accordance to the way the model was trained
            max_new_tokens=1024, # high max toke otherwise output is truncated
            device_map="auto",
        )
        return pipeline, tokenizer

    def generate_response(
        self, question: str, context: list, df_ci_geo: pd.DataFrame, chunk_id: int
    ):
    
        rendered_prompt = template.render(
            context=context,
            question=question,
        )
        print(f"Generating response for chunk_id: {chunk_id} ...")

        sequences = self.pipeline(
            rendered_prompt,  # jinja template
            max_new_tokens=1024, # use default to not truncate the LLM response
            do_sample=True,
            num_beams=1,  # select token based on probability distribution over entire model’s vocabulary
            # top_k=10,
            # top_p=0.5,
            temperature=0.2,
            # num_return_sequences=1,
            eos_token_id=self.tokenizer.eos_token_id,
            return_full_text=False,  # allow bullet point answers
        )
        # Extracting and returning the generated text
        return sequences

In [ ]:

PARSED_TEXT_DIR = "../" + s.settings.PATH_DATA + "parsed_documents/"


## call language model for recognition of CI and geolocation
## see for more info: https://spacy.io/usage/rule-based-matching#entityruler
## NOTE EntityRuler is hidden inside .add_pipe()
# NOTE Creating new entity (CI_TYPE) solves the issue that FAC entities (buildings, airports, highways, bridges, etc.) refer only to the name of the facility (e.g. A76, Ahrtalbahn)
config = {"spans_key": None, "annotate_ents": True, "overwrite": False}
try:
    ruler = nlp.add_pipe("span_ruler", config=config)
    ruler.from_disk("../ner_patterns.jsonl")
except ValueError:
    print("SpanRuler already exists in pipeline.")
    ruler = nlp.get_pipe("span_ruler")
    ruler.from_disk("../ner_patterns.jsonl")


df_responses = pd.DataFrame(
    columns=[
        "chunk_id",
        "infrastructure_type",
        "damage",
        "societal_impact",
        "economic_impact",
        "location",
        # "time",
        # "duration",
        "confidence",
        "confidence_explanation",
        # "text_snippet",
    ]
)

## init LLM pipeline
decoder_model = DecoderModel()



## iterate over documents
for i, filename in enumerate(glob.glob(str(Path(PARSED_TEXT_DIR, "*cleaned.md")))):

    no_documents = len(glob.glob(str(Path(PARSED_TEXT_DIR, "*cleaned.md"))))
    filepath = Path(filename)
    filename_stem = filepath.stem


    print(f"\n\n ######## -------- Processing document [{i+1}/{no_documents}]: {filepath.name} -------- ######## \n")

    ## extract authors, publication year and title 
    citation_pattern = r"(.*?)(\d{4})(.*)" # split at first occurrence of year
    try:
        authors, year, title = re.findall(citation_pattern, filename_stem)[0]
        citation = f"{authors} {year}"
    except AttributeError as e:
        print(f"Could not extract citation from title: {e}")
        citation = filename_stem


    print(f"\n ######## -------- Getting geolocation of CI assets -------- ######## \n")
    ## Create New entity for transport infrastructure and apply it on any doc

    ## TODO make as pydantic class with fixed attributes
    df_ci_geo = pd.DataFrame(
        columns=[
            "chunk_id",
            "ci_entity",
            "ci_entity_label",
            "geo_entity",
            "geo_entity_label",
            "token_distance",
        ]
    )

    ## load doc
    loader = DoclingLoader(filepath)  # use chunks from Docling.Loader
    doc = loader.load()
    doc = doc[5:15]

    ## get most likely geolocation for each CI entity based on distance
    for i, chunk in enumerate(doc):
        nlp_chunk = nlp(chunk.page_content)
        all_ents = [ent for ent in nlp_chunk.ents]
        ci_type_ents = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE", "FAC"]]
        ci_type_ents_info = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE"]]
        fac_ents_info = [ent for ent in nlp_chunk.ents if ent.label_ in ["FAC"]]

        # check if chunk contains CI_TYPE entities
        if len(ci_type_ents) > 0:
            print(f"\nChunk [{i}], No. CI_TYPE and FAC entities: {len(ci_type_ents)}")
            # print(
            #     f"Contains following entities for CI_TYPE: {ci_type_ents_info}, FAC: {fac_ents_info}"
            # )
            # print(f"Chunk text [{i}]:", chunk.page_content)

            # iterate over all entities within chunk
            for ent_idx in range(len(all_ents)):
                # when entity is CI_TYPE or FAC (i.e. buidling, airports, highways) do following ...
                if all_ents[ent_idx].label_ in ["CI_TYPE", "FAC"]:
                    ci_idx = ent_idx

                    ## .. calculate distances between CI_TYPE entity and  all GEO entities in chunk based on index position
                    distance_list = []
                    idx_in_chunk = []
                    try:
                        for ent_idx in range(len(all_ents)):
                            # TODO calc distances between CI_TYPE ~ GEO entities based on word numbers and not entities
                            if all_ents[ent_idx].label_ in ["GPE", "LOC"]:
                                geo_idx = ent_idx
                                dist_ent_pair = np.abs(ci_idx - geo_idx)
                                distance_list.append(dist_ent_pair)
                                idx_in_chunk.append((ent_idx))
                                closest_pair_idx = np.argmin(
                                    distance_list
                                )  # idx of closest GEO entity
                                distance_closest_pair = distance_list[closest_pair_idx]

                        threshold = 5  # max token distance between CI_TYPE and GEO entity
                        if distance_closest_pair > threshold:
                            print(
                                f""" Token distance between CI_TYPE/FAR and next GEO entity is {distance_closest_pair} and thus larger than the allowed distance of {threshold} tokens """
                            )
                            continue
                        else:
                            pass
                            # print(
                            #     f"""  Closest GEO entity to CI_TYPE/FAC entity "{all_ents[ci_idx]}" is "{all_ents[idx_in_chunk[closest_pair_idx]]}" at distance {distance_closest_pair}"""
                            # )

                        ## write as dict entry incl chunk_id, ci_entity, geo_entity, distance
                        result_dict = {
                            "chunk_id": i,
                            "ci_entity": all_ents[ci_idx].text,
                            "ci_entity_label": all_ents[ci_idx].label_,
                            "geo_entity": all_ents[idx_in_chunk[closest_pair_idx]].text,
                            "geo_entity_label": all_ents[
                                idx_in_chunk[closest_pair_idx]
                            ].label_,
                            "token_distance": distance_closest_pair,
                        }
                        df_ci_geo = pd.concat(
                            [df_ci_geo, pd.DataFrame([result_dict])], ignore_index=True
                        )

                    except IndexError:
                        # print("No GEO entities found in this chunk.")
                        continue
                    # print("\nidx_in_chunk, closest pair idx", idx_in_chunk, closest_pair_idx)

                    # spacy.displacy.render(
                    #     nlp_chunk, style="ent",
                    #     options={"ents": ["CI_TYPE", "GPE", "LOC", "FAC"], "colors": {"CI_TYPE": "violet"}}
                    # )

            else:
                print("\nNo CI_TYPE or FAC entities found in this chunk.")
                continue


    print(f"\n  #############  -------- Text-2-Data: {filepath.name} -------- #############  \n")

    ## apply decoder on each chunk
    ## TODO replace iteration by loading entire document and use recursive chunking from langchain
    for j, chunk in enumerate(doc):

        if df_ci_geo.loc[df_ci_geo["chunk_id"] == j].empty:
            continue

        context = [
            {
                "text": chunk.page_content,
                "citation": citation,
                "ci_locations": df_ci_geo.loc[df_ci_geo["chunk_id"] == j],
            },  # TODO use author names or Primary keys from DB later
        ]

        response = decoder_model.generate_response(
            question=question, context=context, df_ci_geo=df_ci_geo, chunk_id=j
        )

        ## postprocess response
        resp = response[0]["generated_text"].replace("\n", "")
        try:
            resp = (resp.split("]")[0] + "]")  # remove potential text outside of json object
            df_resp = pd.read_json(StringIO(resp))
            df_resp["chunk_id"] = j  # add chunk id as identifier
            df_resp["citation"] = citation  # add citation info
            df_responses = pd.concat([df_responses, df_resp], ignore_index=True)
        except ValueError as e:
            print(f"Cannot add response to output dataframe: {e}, \n{resp}")



    # clean up after each document
    gc.collect()
    torch.cuda.empty_cache()

./huggingface_mirror/hub/
Using locally saved model from ./huggingface_mirror/hub/


`low_cpu_mem_usage` was None, now default to True since model is quantized.
Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.55s/it]
Device set to use cuda:0




 ######## -------- Processing document [1/3]: Mohr 2022 A multi-disciplinary analysis of the exceptional flood event of July 2021 in central Europe - Part 1 Event desciption and analysis_cleaned.md -------- ######## 


 ######## -------- Getting geolocation of CI assets : Mohr 2022 A multi-disciplinary analysis of the exceptional flood event of July 2021 in central Europe - Part 1 Event desciption and analysis_cleaned.md -------- ######## 



2025-12-04 21:49:42,675 - INFO - Going to convert document batch...
2025-12-04 21:49:42,675 - INFO - Initializing pipeline for SimplePipeline with options hash 4cc01982ae99b46a2a63fcda46c47c35
2025-12-04 21:49:42,676 - INFO - Processing document Mohr 2022 A multi-disciplinary analysis of the exceptional flood event of July 2021 in central Europe - Part 1 Event desciption and analysis_cleaned.md
2025-12-04 21:49:42,860 - INFO - Finished converting document Mohr 2022 A multi-disciplinary analysis of the exceptional flood event of July 2021 in central Europe - Part 1 Event desciption and analysis_cleaned.md in 0.19 sec.
Token indices sequence length is longer than the specified maximum sequence length for this model (609 > 512). Running this sequence through the model will result in indexing errors



Chunk [0], No. CI_TYPE and FAC entities: 3
  Closest GEO entity to CI_TYPE/FAC entity "railways" is "Rhine" at distance 2
  Closest GEO entity to CI_TYPE/FAC entity "roads" is "Rhine" at distance 3
  Closest GEO entity to CI_TYPE/FAC entity "bridges" is "Rhine" at distance 4

No CI_TYPE or FAC entities found in this chunk.

Chunk [6], No. CI_TYPE and FAC entities: 2
No GEO entities found in this chunk.
No GEO entities found in this chunk.

No CI_TYPE or FAC entities found in this chunk.

  #############  -------- Text-2-Data: Mohr 2022 A multi-disciplinary analysis of the exceptional flood event of July 2021 in central Europe - Part 1 Event desciption and analysis_cleaned.md -------- #############  

Generating response for chunk_id: 0 ...


 ######## -------- Processing document [2/3]: Koks et al 2022 Brief communication_cleaned.md -------- ######## 


 ######## -------- Getting geolocation of CI assets : Koks et al 2022 Brief communication_cleaned.md -------- ######## 



...


Chunk [0], No. CI_TYPE and FAC entities: 10
  Closest GEO entity to CI_TYPE/FAC entity "road" is "Germany" at distance 1
  Closest GEO entity to CI_TYPE/FAC entity "railway" is "Germany" at distance 2
  Closest GEO entity to CI_TYPE/FAC entity "motorways" is "Hauser" at distance 4
  Closest GEO entity to CI_TYPE/FAC entity "bridges" is "the Ahr valley" at distance 2
  Closest GEO entity to CI_TYPE/FAC entity "bridges" is "Rhineland-Palatinate" at distance 2
  Closest GEO entity to CI_TYPE/FAC entity "roads" is "the Ahr valley" at distance 2
  Closest GEO entity to CI_TYPE/FAC entity "bridges" is "the Ahr valley" at distance 1
  Closest GEO entity to CI_TYPE/FAC entity "A1" is "the Ahr valley" at distance 4
  Closest GEO entity to CI_TYPE/FAC entity "motorway" is "the Ahr valley" at distance 5
 Token distance between CI_TYPE/FAR and next GEO entity is 14 and thus larger than the allowed distance of 5 tokens 

No CI_TYPE or FAC entities found in this chunk.

Chunk [1], No. CI_TYPE and F

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Generating response for chunk_id: 9 ...


 ######## -------- Processing document [3/3]: Korzilius 2021 Nach der Flut_cleaned.md -------- ######## 


 ######## -------- Getting geolocation of CI assets : Korzilius 2021 Nach der Flut_cleaned.md -------- ######## 



2025-12-04 21:56:36,567 - INFO - Going to convert document batch...
2025-12-04 21:56:36,568 - INFO - Initializing pipeline for SimplePipeline with options hash 4cc01982ae99b46a2a63fcda46c47c35
2025-12-04 21:56:36,568 - INFO - Processing document Korzilius 2021 Nach der Flut_cleaned.md
2025-12-04 21:56:36,601 - INFO - Finished converting document Korzilius 2021 Nach der Flut_cleaned.md in 0.04 sec.
Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors



Chunk [0], No. CI_TYPE and FAC entities: 1
No GEO entities found in this chunk.

No CI_TYPE or FAC entities found in this chunk.

Chunk [9], No. CI_TYPE and FAC entities: 1
  Closest GEO entity to CI_TYPE/FAC entity "Hospital" is "Erftstadt" at distance 1

No CI_TYPE or FAC entities found in this chunk.

  #############  -------- Text-2-Data: Korzilius 2021 Nach der Flut_cleaned.md -------- #############  

Generating response for chunk_id: 9 ...
Cannot add response to output dataframe: Unmatched ''"' when when decoding 'string', 
    [        {            "infrastructure_type": "Hospital",            "damage": "Zerstörung",            "societal_impact": "NAN",            "economic_impact": "NAN",            "location": "Erftstadt",            "confidence_explanation": "[The text does not provide any information about societal or economic consequences of the hospital failure.]
